We perform equal weights + risk parity + CVaR approaches to test strength of these strategies during dynamic market conditions. We test for both generic market conditions as well as (very) volatile downturns.

The goal is to minimize tail risk whilst attempting to minimize the impact on general returns / sharpe ratio.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from scipy.stats import multivariate_normal, norm

snp = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "snp500.csv", index_col=1)
china = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "china_transformed.csv", index_col=1).reset_index()
em = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "emerging_markets.csv", index_col=1)
gold = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "gold.csv", index_col=1)
india = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "india.csv", index_col=1)
mscieurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "msci_europe.csv", index_col=1)
smallcapeurope = pd.read_csv(Path.cwd().parent / "data" / "OHCL" / "small_cap_europe.csv", index_col=1)

labels = ["snp", "china", "em", "gold", "india", "mscieurope", "smallcapeurope"]
dfs = [snp, china, em, gold, india, mscieurope, smallcapeurope]

def transform_date(date):
    parts = date.split(' ')
    if len(parts) == 1:
        return date
    return parts[0]

for df in dfs:
    df["Date"] = df["Date"].apply(transform_date)

china

date_sets = [set(df["Date"]) for df in dfs]

# Find the intersection of all date sets
common_dates = set.intersection(*date_sets)

# Filter each DataFrame to include only common dates
filtered_dfs = [df[df["Date"].isin(common_dates)] for df in dfs]

#for df in dfs:
#    df["Date"] = df["Date"].apply(pd.to_datetime)

#print(filtered_dfs[0].iloc[0:90]["Date"].values)

In [2]:

N = len(filtered_dfs)
T = len(filtered_dfs[0]) - 1

returns_all = np.empty((T, N))
sorted_returns_all = np.empty((T, N))
for i, df in enumerate(filtered_dfs):
    returns = df["Close"].pct_change().values[1:]
    returns_all[:, i] = returns
    sorted_returns_all[:, i] = sorted(returns)

all_dates = filtered_dfs[0]["Date"].values[1:]

In [6]:
def perform_validation(w, validation_data):
    # validation data is shape (T, N)
    
    # perform metrics
    total_return = validation_data @ w # shape (T,)
    cumulative_returns = np.cumprod(total_return + 1) - 1
    risk_free = 0.0
    std = np.std(total_return)
    downside_std = np.std(total_return[total_return > 0])
    sharpe = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * std)
    sortino = (255 * np.mean(total_return) - risk_free) / (np.sqrt(255) * downside_std)    

    alpha = 0.10

    var_alpha = np.quantile(total_return, alpha)
    cvar = total_return[total_return <= var_alpha].mean()
    mean_return = np.mean(total_return)
    return var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
    

In [7]:
import tensorflow as tf

# train weights
alpha = 0.05   # tail probability for CVaR
lambda_ = 0.5  # CVaR penalty
learning_rate = 0.01
gamma = 0.02
num_steps = 2000
beta = 0.1 # penalty volatility

def gradient(train_data, cov):

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))
            
            # portfolio variance
            portfolio_variance = tf.squeeze(tf.transpose(w_col) @ cov_tf @ w_col)

            objective = (
                -(tf.reduce_mean(portfolio_returns) - lambda_ * cvar)
                + beta * portfolio_variance
            )

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        if step % 200 == 0:
            print(f"Step {step}: objective = {-objective.numpy():.6f}")

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()
    return optimal_w
    
def gradient_risk_parity(cov):

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)

    optimizer = tf.keras.optimizers.Adam(learning_rate)
    cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)
            
            # portfolio variance
            portfolio_variance = tf.squeeze(tf.transpose(w_col) @ cov_tf @ w_col)

            marginal = cov_tf @ w_col
            risk_contrib = w_col * marginal

            risk_parity_target = portfolio_variance / N
            objective = tf.reduce_sum((risk_contrib - risk_parity_target)**2) + gamma*tf.reduce_sum(tf.math.square(w_col))

        # compute gradients
        grads = tape.gradient(objective, [w])
        optimizer.apply_gradients(zip(grads, [w]))

        if step % 200 == 0:
            print(f"Step {step}: objective = {-objective.numpy():.6f}")

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()
    return optimal_w

def gradient_reduced(train_data, cov):

    # train data: (T,N)
    w = tf.Variable(tf.ones([N], dtype=tf.float32) / N)
    t = tf.Variable(0.0, dtype=tf.float32)  # VaR threshold

    optimizer = tf.keras.optimizers.Adam(learning_rate)

    X_tf = tf.convert_to_tensor(train_data, dtype=tf.float32) # (T,N)
    cov_tf = tf.convert_to_tensor(cov, dtype=tf.float32)  # shape (N, N)

    for step in range(num_steps):
        with tf.GradientTape() as tape:
            # enforce sum-to-1 via softmax
            w_normalized = tf.nn.softmax(w)
            w_col = tf.reshape(w_normalized, [N, 1])  # shape (N,1)

            # portfolio returns
            portfolio_returns = tf.matmul(X_tf, tf.reshape(w_col, [-1,1])) # (T)
            losses = -portfolio_returns
            cvar = t + (1 / (1 - alpha)) * tf.reduce_mean(tf.nn.relu(losses - t))

            objective = -(tf.reduce_mean(portfolio_returns) - lambda_ * cvar) + gamma*tf.reduce_sum(tf.math.square(w_col))

        # compute gradients
        grads = tape.gradient(objective, [w, t])
        optimizer.apply_gradients(zip(grads, [w, t]))

        if step % 200 == 0:
            print(f"Step {step}: objective = {-objective.numpy():.6f}")

    # after the optimization loop
    optimal_w = tf.nn.softmax(w).numpy()
    return optimal_w

In [8]:
# run

def select_random_block(returns, block_size):
    n = len(returns)
    lst = list(range(n))
    chosen_idx = np.random.choice(lst)
    while not (block_size < chosen_idx < n - block_size):
        chosen_idx = np.random.choice(n)
    selected_train_data = returns[chosen_idx - block_size : chosen_idx]
    selected_validation_data = returns[chosen_idx : chosen_idx + block_size]
    return selected_train_data, selected_validation_data

def block_bootstrap(n_bootstrap, returns_all):

    sharpes_w_opt= []
    sharpes_w_rp = []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_rp = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_rp = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_rp = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_rp = []
    cvars_w_eqw = []

    for _ in range(n_bootstrap):
        train_data, val_data = select_random_block(returns_all, 252)
        cov = np.cov(train_data, rowvar=False)

        w_opt = gradient(train_data, cov)
        w_rp = gradient_risk_parity(cov)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(w_opt, val_data)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(w_rp, val_data)

        sharpes_w_rp.append(sharpe)
        sortinos_w_rp.append(sortino)
        vars_w_rp.append(var_alpha)
        cvars_w_rp.append(cvar)
        mean_return_w_rp.append(mean_return)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(np.ones(N) / N, val_data)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_rp), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_rp), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_rp), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_rp), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_rp), np.mean(mean_return_w_opt)],
    }, index=["Equal weights", "Risk parity", "Return + CVaR"])

    return df

def rolling_window(window_size, stepsize, validation_size, data):
    # data is (T,N)

    sharpes_w_opt= []
    sharpes_w_rp = []
    sharpes_w_eqw = []

    mean_return_w_opt = []
    mean_return_w_rp = []
    mean_return_w_eqw = []

    sortinos_w_opt = []
    sortinos_w_rp = []
    sortinos_w_eqw = []

    vars_w_opt = []
    vars_w_rp = []
    vars_w_eqw = []

    cvars_w_opt = []
    cvars_w_rp = []
    cvars_w_eqw = []
    T, N = data.shape
    weights_opt = []
    weights_rp = []
    for idx in range(0, T - window_size - validation_size, stepsize):
        train_data = data[idx:idx+window_size]
        val_data = data[idx+window_size:idx+window_size+validation_size]
        cov = np.cov(train_data, rowvar=False)

        w_opt = gradient_reduced(train_data, cov)
        w_rp = gradient_risk_parity(cov)

        print('w opt:', w_opt)
        print('w rp:', w_rp)

        weights_opt.append(w_opt)
        weights_rp.append(w_rp)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(w_opt, val_data)

        sharpes_w_opt.append(sharpe)
        sortinos_w_opt.append(sortino)
        vars_w_opt.append(var_alpha)
        cvars_w_opt.append(cvar)
        mean_return_w_opt.append(mean_return)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(w_rp, val_data)

        sharpes_w_rp.append(sharpe)
        sortinos_w_rp.append(sortino)
        vars_w_rp.append(var_alpha)
        cvars_w_rp.append(cvar)
        mean_return_w_rp.append(mean_return)

        (
            var_alpha, cvar, sharpe, sortino, cumulative_returns, mean_return
        ) = perform_validation(np.ones(N) / N, val_data)

        sharpes_w_eqw.append(sharpe)
        sortinos_w_eqw.append(sortino)
        vars_w_eqw.append(var_alpha)
        cvars_w_eqw.append(cvar)
        mean_return_w_eqw.append(mean_return)

    df = pd.DataFrame({
        "Sharpes": [np.mean(sharpes_w_eqw), np.mean(sharpes_w_rp), np.mean(sharpes_w_opt)],
        "Sortinos": [np.mean(sortinos_w_eqw), np.mean(sortinos_w_rp), np.mean(sortinos_w_opt)],
        "Vars": [np.mean(vars_w_eqw), np.mean(vars_w_rp), np.mean(vars_w_opt)],
        "Cvars": [np.mean(cvars_w_eqw), np.mean(cvars_w_rp), np.mean(cvars_w_opt)],
        "MeanReturn": [np.mean(mean_return_w_eqw), np.mean(mean_return_w_rp), np.mean(mean_return_w_opt)],
    }, index=["Equal weights", "Risk parity", "Return + CVaR"])

    weights_opt = np.array(weights_opt)
    weights_rp = np.array(weights_rp)

    print(weights_opt.shape)

    return df, np.mean(weights_opt, axis=0), np.mean(weights_rp, axis=0)



In [9]:
df, weights_opt, weights_rp = rolling_window(
    window_size=252*2,
    stepsize=30,
    validation_size=252,
    data=returns_all
)
df

print('mean weights optimized:', weights_opt)
print('mean weights risk parity:', weights_rp)

Step 0: objective = -0.004332
Step 200: objective = -0.003232
Step 400: objective = -0.003232
Step 600: objective = -0.003232
Step 800: objective = -0.003232
Step 1000: objective = -0.003232
Step 1200: objective = -0.003232
Step 1400: objective = -0.003232
Step 1600: objective = -0.003232
Step 1800: objective = -0.003232
Step 0: objective = -0.002857
Step 200: objective = -0.002857
Step 400: objective = -0.002857
Step 600: objective = -0.002857
Step 800: objective = -0.002857
Step 1000: objective = -0.002857
Step 1200: objective = -0.002857
Step 1400: objective = -0.002857
Step 1600: objective = -0.002857
Step 1800: objective = -0.002857
w opt: [0.15016459 0.12259774 0.14088295 0.15251325 0.13363686 0.14659537
 0.15360917]
w rp: [0.14287    0.14287175 0.14285295 0.14285785 0.14284307 0.1428522
 0.1428522 ]
Step 0: objective = -0.004238
Step 200: objective = -0.003134
Step 400: objective = -0.003134
Step 600: objective = -0.003134
Step 800: objective = -0.003134
Step 1000: objective = -

Run once more, using entire dataset

In [11]:
training_pct = 0.7
idx = int(training_pct*T)

training_set_entire = returns_all[:idx,:]
validation_set_entire = returns_all[idx:,:]

cov = np.cov(training_set_entire, rowvar=False)
w_opt = gradient_reduced(training_set_entire, cov)
w_rp = gradient_risk_parity(cov)

(
    var_alpha, cvar, sharpe, sortino, cumulative_returns_opt, mean_return
) = perform_validation(w_opt, validation_set_entire)

print('OPTIMIZED WEIGHTS STATS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)

(
    var_alpha, cvar, sharpe, sortino, cumulative_returns_rp, mean_return
) = perform_validation(w_rp, training_set_entire)

print()
print('RISK PARITY STATS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)

(
    var_alpha, cvar, sharpe, sortino, cumulative_returns_eqw, mean_return
) = perform_validation(np.ones(N) / N, training_set_entire)

print()
print('EQUAL WEIGHTS STATS')
print('var:', var_alpha)
print('cvar:', cvar)
print('sharpe:', sharpe)
print('sortino:', sortino)
print('mean return:', mean_return)

print()
print('optimized weights:', w_opt)
print('risk parity weights:', w_rp)

fig = go.Figure()
L = list(range(len(validation_set_entire)))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_opt, name="optimized weights"))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_rp, name="risk parity weights"))
fig.show()


Step 0: objective = -0.004094
Step 200: objective = -0.002960
Step 400: objective = -0.002960
Step 600: objective = -0.002960
Step 800: objective = -0.002960
Step 1000: objective = -0.002960
Step 1200: objective = -0.002960
Step 1400: objective = -0.002960
Step 1600: objective = -0.002960
Step 1800: objective = -0.002960
Step 0: objective = -0.002857
Step 200: objective = -0.002857
Step 400: objective = -0.002857
Step 600: objective = -0.002857
Step 800: objective = -0.002857
Step 1000: objective = -0.002857
Step 1200: objective = -0.002857
Step 1400: objective = -0.002857
Step 1600: objective = -0.002857
Step 1800: objective = -0.002857
OPTIMIZED WEIGHTS STATS
var: -0.006032235755893444
cvar: -0.010700151303303526
sharpe: 1.6171918538482555
sortino: 2.555942477348651
mean return: 0.0006213304502896785

RISK PARITY STATS
var: -0.008907919140814962
cvar: -0.01533025409237657
sharpe: 0.4342712013149262
sortino: 0.7033863189516916
mean return: 0.0002252145769865999

EQUAL WEIGHTS STATS
va

In [76]:
def sample(sorted_returns, amount):
    res = []
    n = len(sorted_returns)
    for _ in range(amount):
        u = np.random.uniform()
        for i,val in enumerate(sorted_returns):
            if i >= n*u:
                res.append(val)
                break
    return res

def sample_combined(sorted_returns, amount, weights):
    # weights: (N)
    # sorted returns: (T, N) array
    final = np.zeros_like(sorted_returns)
    for i in range(sorted_returns.shape[1]):
        final[:,i] = sample(sorted_returns[:,i], amount)
    
    return final @ weights

In [77]:
df = block_bootstrap(200, returns_all)
df

Step 0: objective = -0.003101
Step 200: objective = 0.000059
Step 400: objective = 0.000129
Step 600: objective = 0.000149
Step 800: objective = 0.000159
Step 1000: objective = 0.000162
Step 1200: objective = 0.000165
Step 1400: objective = 0.000162
Step 1600: objective = 0.000164
Step 1800: objective = 0.000169
Step 0: objective = -0.000000
Step 200: objective = -0.000000
Step 400: objective = -0.000000
Step 600: objective = -0.000000
Step 800: objective = -0.000000
Step 1000: objective = -0.000000
Step 1200: objective = -0.000000
Step 1400: objective = -0.000000
Step 1600: objective = -0.000000
Step 1800: objective = -0.000000
Step 0: objective = -0.002597
Step 200: objective = 0.000866
Step 400: objective = 0.001011
Step 600: objective = 0.001039
Step 800: objective = 0.001048
Step 1000: objective = 0.001052
Step 1200: objective = 0.001058
Step 1400: objective = 0.001060
Step 1600: objective = 0.001061
Step 1800: objective = 0.001062
Step 0: objective = -0.000000
Step 200: objective

,Sharpes,Sortinos,Vars,Cvars,MeanReturn
Equal weights,1.051824,1.738873,-0.007703,-0.013041,0.000412
Risk parity,1.039199,1.717977,-0.007680,-0.012998,0.000405
Return + CVaR,0.583604,0.900634,-0.008890,-0.014768,0.000243


Test bear market conditions for all approaches

In [ ]:

val_returns_total = []
train_returns_total = []

for df in filtered_dfs:
    df_datetime = df.copy()
    df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
    train = df_datetime[
        (df_datetime["Date"] >= pd.to_datetime("2024-01-01"))
        & (df_datetime["Date"] <= pd.to_datetime("2025-01-01"))]
    val = df_datetime[
        (df_datetime["Date"] >= pd.to_datetime("2025-01-01"))
        & (df_datetime["Date"] <= pd.to_datetime("2025-04-01"))
    ]

    train_returns = train["Close"].pct_change().values[1:]
    val_returns = val["Close"].pct_change().values[1:]

    print(len(train_returns))
    print(len(val_returns))
    print()

    val_returns_total.append(val_returns)
    train_returns_total.append(train_returns)
    
train_returns_total = np.array(train_returns_total).T # shape (num_days, N = num_assets)
val_returns_total = np.array(val_returns_total).T

cov = np.cov(train_returns_total, rowvar=False)

w_opt = gradient_reduced(train_returns_total, cov)
w_rp = gradient_risk_parity(cov)

print('chosen weights optimal:', w_opt)
print('chosen weights risk parity:', w_rp)

(
    var_alpha_opt, cvar_opt, sharpe_opt, sortino_opt, cumulative_returns_opt, mean_return_opt
) = perform_validation(w_opt, val_returns_total)

(
    var_alpha_rp, cvar_rp, sharpe_rp, sortino_rp, cumulative_returns_rp, mean_return_rp
) = perform_validation(w_rp, val_returns_total)

(
    var_alpha_eqw, cvar_eqw, sharpe_eqw, sortino_eqw, cumulative_returns_eqw, mean_return_eqw
) = perform_validation(np.ones(N) / N, val_returns_total)

df = pd.DataFrame({
    "Sharpe": [sharpe_opt, sharpe_eqw, sharpe_rp],
    "Sortino": [sortino_opt, sortino_eqw, sortino_rp],
    "Var": [var_alpha_opt, var_alpha_eqw, var_alpha_rp],
    "Cvar": [cvar_opt, cvar_eqw, cvar_rp],
    "Mean return": [mean_return_opt, mean_return_eqw, mean_return_rp],
}, index=["Optimized weights", "Equal weights", "Risk parity weights"])
df


249
63

249
63

249
63

249
63

249
63

249
63

249
63

Step 0: objective = -0.003396
Step 200: objective = -0.002446
Step 400: objective = -0.002446
Step 600: objective = -0.002446
Step 800: objective = -0.002446
Step 1000: objective = -0.002446
Step 1200: objective = -0.002446
Step 1400: objective = -0.002446
Step 1600: objective = -0.002445
Step 1800: objective = -0.002446
Step 0: objective = -0.002857
Step 200: objective = -0.002857
Step 400: objective = -0.002857
Step 600: objective = -0.002857
Step 800: objective = -0.002857
Step 1000: objective = -0.002857
Step 1200: objective = -0.002857
Step 1400: objective = -0.002857
Step 1600: objective = -0.002857
Step 1800: objective = -0.002857
chosen weights optimal: [0.17072469 0.10815918 0.1384753  0.17181906 0.14737406 0.13307038
 0.13037725]
chosen weights risk parity: [0.14288683 0.14286682 0.14288683 0.14288683 0.1428242  0.1428242
 0.1428242 ]


,Sharpe,Sortino,Var,Cvar,Mean return
Optimized weights,0.582074,1.208277,-0.007756,-0.010640,0.000214
Equal weights,0.628597,1.289585,-0.007833,-0.010334,0.000231
Risk parity weights,0.628562,1.289514,-0.007832,-0.010334,0.000231


In [96]:
fig = go.Figure()
L = list(range(len(cumulative_returns_opt)))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_rp, name="risk parity weights"))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_opt, name="optimized weights"))
fig.show()


In [13]:

val_returns_total = []
train_returns_total = []

for df in filtered_dfs:
    df_datetime = df.copy()
    df_datetime["Date"] = pd.to_datetime(df_datetime["Date"])
    train = df_datetime[
        (df_datetime["Date"] >= pd.to_datetime("2019-02-05"))
        & (df_datetime["Date"] <= pd.to_datetime("2020-02-12"))]
    val = df_datetime[
        (df_datetime["Date"] >= pd.to_datetime("2020-02-12"))
        & (df_datetime["Date"] <= pd.to_datetime("2020-04-03"))
    ]

    train_returns = train["Close"].pct_change().values[1:]
    val_returns = val["Close"].pct_change().values[1:]

    print(len(train_returns))
    print(len(val_returns))
    print()

    val_returns_total.append(val_returns)
    train_returns_total.append(train_returns)
    
train_returns_total = np.array(train_returns_total).T # shape (num_days, N = num_assets)
val_returns_total = np.array(val_returns_total).T

cov = np.cov(train_returns_total, rowvar=False)

w_opt = gradient_reduced(train_returns_total, cov)
w_rp = gradient_risk_parity(cov)

print('chosen weights optimal:', w_opt)
print('chosen weights risk parity:', w_rp)

(
    var_alpha_opt, cvar_opt, sharpe_opt, sortino_opt, cumulative_returns_opt, mean_return_opt
) = perform_validation(w_opt, val_returns_total)

(
    var_alpha_rp, cvar_rp, sharpe_rp, sortino_rp, cumulative_returns_rp, mean_return_rp
) = perform_validation(w_rp, val_returns_total)

(
    var_alpha_eqw, cvar_eqw, sharpe_eqw, sortino_eqw, cumulative_returns_eqw, mean_return_eqw
) = perform_validation(np.ones(N) / N, val_returns_total)

df = pd.DataFrame({
    "Sharpe": [sharpe_opt, sharpe_eqw, sharpe_rp],
    "Sortino": [sortino_opt, sortino_eqw, sortino_rp],
    "Var": [var_alpha_opt, var_alpha_eqw, var_alpha_rp],
    "Cvar": [cvar_opt, cvar_eqw, cvar_rp],
    "Mean return": [mean_return_opt, mean_return_eqw, mean_return_rp],
}, index=["Optimized weights", "Equal weights", "Risk parity weights"])
df


252
37

252
37

252
37

252
37

252
37

252
37

252
37

Step 0: objective = -0.003276
Step 200: objective = -0.002289
Step 400: objective = -0.002289
Step 600: objective = -0.002289
Step 800: objective = -0.002289
Step 1000: objective = -0.002289
Step 1200: objective = -0.002290
Step 1400: objective = -0.002291
Step 1600: objective = -0.002291
Step 1800: objective = -0.002291
Step 0: objective = -0.002857
Step 200: objective = -0.002857
Step 400: objective = -0.002857
Step 600: objective = -0.002857
Step 800: objective = -0.002857
Step 1000: objective = -0.002857
Step 1200: objective = -0.002857
Step 1400: objective = -0.002857
Step 1600: objective = -0.002857
Step 1800: objective = -0.002857
chosen weights optimal: [0.1651642  0.09643536 0.1311008  0.16436002 0.14088394 0.14744465
 0.15461107]
chosen weights risk parity: [0.14285715 0.14285715 0.14285715 0.14285715 0.14285715 0.14285715
 0.14285715]


,Sharpe,Sortino,Var,Cvar,Mean return
Optimized weights,-3.573863,-7.084715,-0.038750,-0.056981,-0.006045
Equal weights,-3.476657,-7.111435,-0.038604,-0.055634,-0.005743
Risk parity weights,-3.476657,-7.111435,-0.038604,-0.055634,-0.005743


In [98]:
fig = go.Figure()
L = list(range(len(cumulative_returns_opt)))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_eqw, name="equal weights"))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_rp, name="risk parity weights"))
fig.add_trace(go.Scatter(x=L, y=cumulative_returns_opt, name="optimized weights"))
fig.show()
